In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from tqdm import tqdm
import json
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import EarlyStopping, print_exams
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datetime import datetime
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from tqdm import tqdm

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

class HANADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['seq_a'] + '<eos>' + DataFrame['seq_b'] + '<eos>' + DataFrame['seq_c'] + '<eos>' + DataFrame['seq_d'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [4]:
## read complete information
train_data = pd.read_csv('./train_data.csv')
test_data = pd.read_csv('./test_data.csv')
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)

In [9]:
## remove duplicated row and mean HI_Dist
HANA_data_filt1 = train_data.groupby(['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'Type': 'first', 'label': 'mean'}) \
        .reset_index()
## remove PassCat = 'BOTH'
HANA_data_filt2 = HANA_data_filt1[(HANA_data_filt1['serumPassCat'] != 'BOTH') &
                              (HANA_data_filt1['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
HANA_data_filt3 = HANA_data_filt2.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

train_data_final = HANA_data_filt3.copy()

In [ ]:
train_dataset = HANADataset(train_data_final)
valid_dataset = HANADataset(valid_data)
test_dataset = HANADataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [13]:
# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=28
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5

device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model = model.to(device)

# 计数：DP时用 .module
print("Number of parameters: %e" % count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

Number of parameters: 1.135435e+06


In [14]:
# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)


progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=20, save_dir='/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/trained_model/1.2_HANA_model/')
# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt").to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls_valid = []
    reference_ls_valid = []
    logits_ls_valid = []
    loss_ls_valid = []
    model.eval()
    with torch.no_grad():
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt").to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls_valid.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls_valid.extend(logits.squeeze().tolist())
            reference_ls_valid += batch_label.tolist()
    valid_mae, valid_mse, valid_pearson, valid_spearman, valid_R2 = print_exams(reference_ls_valid, prediction_ls_valid, print_result=False)

    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    prediction_ls_test = []
    reference_ls_test = []
    logits_ls_test = []
    loss_ls_test = []
    model.eval()
    with torch.no_grad():
        for batch_seq, batch_label in test_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt").to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls_test.append(logits)
            loss_ls_test.append(loss.item())
            prediction_ls_test.extend(logits.squeeze().tolist())
            reference_ls_test.extend(batch_label.tolist())
    
    test_mae, test_mse, test_pearson, test_spearman, test_R2 = print_exams(reference_ls_test, prediction_ls_test, print_result=False)


    ## 将epoch信息写入log.txt
    with open('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/trained_model/1.2_HANA_model/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(
            f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.5f}, valid MAE: {valid_mae:.5f}, valid MSE: {valid_mse:.5f}, "
            f"valid Pearson: {valid_pearson.statistic:.5f}, valid Spearman: {valid_spearman.statistic:.5f}, valid R2: {valid_R2:.5f}, test MAE: {test_mae:.5f}, test MSE: {test_mse:.5f}, "
            f"test Pearson: {test_pearson.statistic:.5f}, test Spearman: {test_spearman.statistic:.5f}, test R2: {test_R2:.5f}\n")

  1%|          | 100/16000 [00:20<48:35,  5.45it/s] 

train loss : 2.3717586889863016
Validation MSE decrease (inf --> 4.945859).  Saving model ...


  1%|▏         | 200/16000 [01:04<48:22,  5.44it/s]   

train loss : 1.7312143036723138
EarlyStopping counter: 1 out of 20


  2%|▏         | 300/16000 [01:47<47:56,  5.46it/s]   

train loss : 1.7450756403803824
Validation MSE decrease (4.945859 --> 4.900929).  Saving model ...


KeyboardInterrupt: 

In [11]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in valid_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE:  0.6779233298936067
MSE:  0.7794392907296332
pearson correlation:  PearsonRResult(statistic=0.8926168447273077, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8651883949093491, pvalue=0.0)
R2_score:  0.794365801944003


In [8]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in test_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE: 0.61797
MSE: 0.70557
pearson correlation: 0.90011
spearman correlation: 0.86934
R2_score: 0.80900


(0.6179685946287751,
 0.7055740593939708,
 PearsonRResult(statistic=0.9001129952198043, pvalue=0.0),
 SignificanceResult(statistic=0.8693438555608725, pvalue=0.0),
 0.8090043627200303)

In [7]:
from bio_tokenizer import BioTokenizer
from utilities import print_exams
import torch

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []

tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')
device = torch.device('cuda:1')
model = torch.load('../../../trained_model/1.2_HANA_model_new/2025-07-19_10-53-19.pth', weights_only=False, map_location=device)
model.eval()
for batch_seq, batch_label in valid_loader:
    batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
    batch_input = batch_input.to(device)
    batch_label = batch_label.to(device)
    with torch.no_grad():
        outputs = model(**batch_input, labels=batch_label)
    logits = outputs.logits
    loss = outputs.loss

    logits_ls.append(logits)
    loss_ls_valid.append(loss.item())
    prediction_ls += logits.tolist()
    prediction_ls_final = []
    for sublist in prediction_ls:
        for element in sublist:
            prediction_ls_final.append(element)
    reference_ls += batch_label.tolist()

print_exams(prediction_ls_final, reference_ls)

MAE: 0.61789
MSE: 0.68847
pearson correlation: 0.90104
spearman correlation: 0.86845
R2_score: 0.81117


(0.617890420807107,
 0.6884695665380479,
 PearsonRResult(statistic=0.9010370210691954, pvalue=0.0),
 SignificanceResult(statistic=0.8684494635969174, pvalue=0.0),
 0.81116525754609)